## 0. Setup

In [1]:
# set libraries to refresh
%load_ext autoreload
%autoreload 2

In [2]:
# general
from pathlib import Path

import geopandas as gpd

# for plotting and coloring
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from tqdm.notebook import tqdm

gpd.options.io_engine = "pyogrio"

In [3]:
from gridsample.utils import create_gmap_links, save_shapefiles
# from gridsample.mapping import create_interactive_map

In [4]:
from utils import (
    download_VIDA_rooftops_data_by_s2,
    generate_colormap,
    get_matched_rooftop_centroids_from_s2_file,
    get_s2_cell_ids,
    s2_cell_ids_to_shapes_gdf,
)

In [ ]:
ROOT_DIR = Path("../")
DATA_DIR = ROOT_DIR / "data_panel"
RAW_DATA_DIR = DATA_DIR / "01. Raw Data"  # symlinked from IFS folder
CLEANED_DATA_DIR = DATA_DIR / "02. Intermediate Outputs" / "training_phase_1_v1"
OUTPUT_DATA_DIR = DATA_DIR / "03. Final Outputs" / "training_phase_1_v1"
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load MapSolve boundaries

In [ ]:
# get all filepaths that end in `gpkg` inside the RAW_DATA_DIR / 0.1. MapSolve Boundaries
boundaries_dir = RAW_DATA_DIR / "01. MapSolve Boundaries"
gpkg_files_all = list(boundaries_dir.glob("**/*.gpkg"))
gpkg_files_all = [f for f in gpkg_files_all if f.is_file()]
# # drop any with the word "Sub-District" in the filename
# gpkg_files_VTW = [f for f in gpkg_files_all if "Sub-District" not in f.name]
# load all shapes into one gdf
gdf_list = []
for filepath in gpkg_files_all:
    gdf_list.append(gpd.read_file(filepath))
gdf = pd.concat(gdf_list, ignore_index=True).to_crs(4326)

In [ ]:
gdf = gdf.drop_duplicates()

### Fix ad-hoc issues (see quality check notebook)

In [ ]:
# chittoor
gdf.loc[gdf["TV_C"] == 803014, ["SubDist_N", "SubDist_C"]] = ["Tirupati (Urban)", 5383]
# patna
gdf.loc[(gdf["TV_C"] == 801373) & (gdf["Ward_C"] == "3"), "PCA_ID"] = "801373-3"
# MDDS codes
gdf.loc[gdf["TV_C"] == 519923, ["TV_C", "Ward_C", "PCA_ID"]] = [802596, 18, "802596-18"]
gdf.loc[gdf["TV_C"] == 574077, ["TV_C", "Ward_C", "PCA_ID"]] = [
    802918,
    157,
    "802918-157",
]

In [ ]:
gdf.plot(column="State_N", legend=True, figsize=(10, 10), cmap="tab20")

## 2. Load sampled wards data

In [ ]:
# load the merged wards data
sample_df = pd.read_csv(
    CLEANED_DATA_DIR
    / "00. Merge and Quality Checks"
    / "Panel Wards with Quality Checks.csv"
)

In [ ]:
sample_df

### 2.1 Rename and clean both datasets

In [ ]:
rename_dict = {
    "UID": "UID",
    "PCA_ID": "PCA_ID",
    "State_C": "State Code",
    "State_N": "State Name",
    "Dist_C": "District Code",
    "Dist_N": "District Name",
    "SubDist_C": "Subdistrict Code",
    "SubDist_N": "Subdistrict Name",
    "TV_C": "TV Code",
    "TV_N": "TV Name",
    "Ward_C": "Ward Code",
    "TOT_P": "Total Population",
}

gdf = gdf.rename(columns=rename_dict)

In [ ]:
rename_dict = {
    "State": "State Code",
    "State_Name": "State Name",
    "District": "District Code",
    "District_Name": "District Name",
    "Subdistrict": "Subdistrict Code",
    "Subd_Name": "Subdistrict Name",
    "TownVillage": "TV Code",
    "UrbanWardVillage": "Ward Code",
    "WardVillage_Name": "Ward/Village Name",
    "TRU": "Urban/Rural",
    "WardVillage_Pop": "Ward Population",
    "Subd_Pop": "Subdistrict Population",
    "State_Pop": "State Population",
    "WardVillageID": "Complete ID",
}
sample_df = sample_df.rename(columns=rename_dict)

# make State Name heading case instead of all caps
sample_df["State Name"] = sample_df["State Name"].str.title()
sample_df.loc[sample_df["State Name"] == "Nct Of Delhi", "State Name"] = "NCT of Delhi"

In [ ]:
# make relevant codes into floats for both datasets
code_columns = [
    "State Code",
    "District Code",
    "Subdistrict Code",
    "TV Code",
    "Ward Code",
]
for col in code_columns:
    sample_df[col] = sample_df[col].astype(float)
    gdf[col] = gdf[col].astype(float)

In [ ]:
sample_df

## 3. Filter MapSolve boundaries to sampled areas

### 3.2 Drop unnecessary rows 

#### Drop rows with no MapSolve shapes

In [ ]:
sample_df[sample_df["PSU Type"] == "none"]

In [ ]:
filtered_df = sample_df[sample_df["PSU Type"] != "none"]

In [ ]:
filtered_df

In [ ]:
# check bad rows, leave as is though
filtered_df[filtered_df["Delivery State"].str.contains("BAD")]

#### Drop duplicate rows if PSU is TV or Subdistrict

In [ ]:
filtered_df = filtered_df[
    # if the PSU Type is "town_village", then drop any duplicated rows with the same TV Code
    ~(
        (filtered_df["PSU Type"] == "town_village")
        & (filtered_df.duplicated(subset=["TV Code"], keep="first"))
    )
    # similar for subdistrict
    & ~(
        (filtered_df["PSU Type"] == "subdistrict")
        & (filtered_df.duplicated(subset=["Subdistrict Code"], keep="first"))
    )
]

filtered_df.value_counts("PSU Type")

#### Duplicated wards

In [ ]:
filtered_df[filtered_df["PCA_ID"].duplicated(keep=False)]

In [ ]:
print(f"Number of rows before dropping PCA_ID duplicates: {len(filtered_df)}")
print(
    f"Number of rows after dropping PCA_ID duplicates: {len(filtered_df.drop_duplicates(subset=['PCA_ID'], keep='first'))}"
)

In [ ]:
filtered_df = filtered_df.drop_duplicates(subset=["PCA_ID"], keep="first")

#### Check for duplicated `PSU ID`

In [ ]:
filtered_df[filtered_df.duplicated(subset=["PSU ID"], keep=False)]

In [ ]:
filtered_df = filtered_df.drop_duplicates(subset=["PSU ID"], keep="first")
len(filtered_df) == filtered_df["PSU ID"].nunique()

## 4. Merge shapes at the respective PSU level

### Wards

In [ ]:
# PCA IDs to match (TVCode-WardCode)
pca_ids_to_match = filtered_df.loc[filtered_df["PSU Type"] == "ward", "PCA_ID"].unique()
len(pca_ids_to_match)

In [ ]:
wards_gdf = gpd.GeoDataFrame(
    filtered_df[filtered_df["PSU Type"] == "ward"].merge(
        gdf[gdf["PCA_ID"].isin(pca_ids_to_match)],
        on=["PCA_ID"],
        how="left",
        suffixes=("", "_MapSolve"),
    ),
    geometry="geometry",
)
wards_gdf.shape

### Town/Village

In [ ]:
# TV codes to match
tv_codes_to_match = filtered_df.loc[
    filtered_df["PSU Type"] == "town_village", "TV Code"
].unique()
len(tv_codes_to_match)

In [ ]:
tv_gdf = gpd.GeoDataFrame(
    filtered_df[filtered_df["PSU Type"] == "town_village"].merge(
        gdf[gdf["TV Code"].isin(tv_codes_to_match) & gdf["Ward Code"].isna()],
        on=["TV Code"],
        how="left",
        suffixes=("", "_MapSolve"),
    )
)
tv_gdf.shape

In [ ]:
# cut out the ward geometries from the TV geometries so we don't double sample those areas
trimmed_tv_geoms = tv_gdf.difference(wards_gdf.geometry.unary_union)
trimmed_tv_gdf = gpd.GeoDataFrame(
    tv_gdf.drop(columns="geometry").assign(geometry=trimmed_tv_geoms),
    crs=tv_gdf.crs,
)

### Subdistrict

In [ ]:
# subdistrict codes to match
subdistrict_codes_to_match = filtered_df.loc[
    filtered_df["PSU Type"] == "subdistrict", "Subdistrict Code"
].unique()
len(subdistrict_codes_to_match)

In [ ]:
subdistrict_gdf = gpd.GeoDataFrame(
    filtered_df[filtered_df["PSU Type"] == "subdistrict"].merge(
        gdf[
            gdf["Subdistrict Code"].isin(subdistrict_codes_to_match)
            & (gdf["TV Code"].isna())
            & (gdf["Ward Code"].isna())
        ],
        on=["Subdistrict Code"],
        how="left",
        suffixes=("", "_MapSolve"),
    )
)

In [ ]:
# cut out the TV and Ward geometries from the subdistrict geometries so we don't double sample those areas
combined_tv_and_ward_union_shape = wards_gdf.geometry.unary_union.union(
    trimmed_tv_gdf.geometry.unary_union
)
if combined_tv_and_ward_union_shape:
    trimmed_subdistrict_geoms = subdistrict_gdf.difference(
        combined_tv_and_ward_union_shape
    )
else:
    trimmed_subdistrict_geoms = subdistrict_gdf.geometry

trimmed_subdistrict_gdf = gpd.GeoDataFrame(
    subdistrict_gdf.drop(columns="geometry").assign(geometry=trimmed_subdistrict_geoms),
    crs=subdistrict_gdf.crs,
)

### Combine

In [ ]:
# combine all three GeoDataFrames into one
combined_gdf = gpd.GeoDataFrame(
    pd.concat(
        [
            wards_gdf,
            trimmed_tv_gdf,
            trimmed_subdistrict_gdf,
        ],
        ignore_index=True,
    )
)
combined_gdf = combined_gdf.sort_values(
    by=["State Code", "District Code", "Subdistrict Code", "TV Code", "Ward Code"]
).reset_index(drop=True)

In [ ]:
combined_gdf.plot(column="State Name")

### Drop rows duplicated `PSU ID` (could happen)

In [ ]:
duplicated_psu_id_gdf = combined_gdf[
    combined_gdf.duplicated(subset=["PSU ID"], keep=False)
]
duplicated_psu_id_gdf

In [ ]:
duplicated_psu_id_gdf.groupby("PSU ID").plot(alpha=0.5, column="PSU ID", legend=True)

In [ ]:
combined_gdf = combined_gdf.drop_duplicates(subset=["PSU ID"], keep="first")
len(combined_gdf) == combined_gdf["PSU ID"].nunique()

In [ ]:
combined_gdf["UID"] = combined_gdf["UID"].astype(str)

In [ ]:
len(combined_gdf) == combined_gdf["PSU ID"].nunique()

In [ ]:
save_shapefiles(
    combined_gdf,
    OUTPUT_DATA_DIR / "Sampled PSUs",
    "all_sampled_PSUs",
    ["csv", "parquet", "kml"],
)

## 5. Download rooftops

#### Identify S2 cell IDs

In [ ]:
s2_cell_ids = get_s2_cell_ids(combined_gdf)
len(s2_cell_ids)

#### Check if identified cells cover all areas of interest

In [ ]:
s2_cells_gdf = s2_cell_ids_to_shapes_gdf(s2_cell_ids)

In [ ]:
# Does the S2 cell cover the entire area of the boundaries?
uncovered_area = combined_gdf.unary_union.difference(s2_cells_gdf.unary_union).area
print(f"{uncovered_area} square degrees area not covered by an S2 cell")

In [ ]:
# Plot the S2 cells and the boundary
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
combined_gdf.boundary.plot(ax=ax, color="black", linewidth=1)
s2_cells_gdf.plot(ax=ax, facecolor="none", edgecolor="red", alpha=0.7)
plt.title("S2 Cells Level 6 Coverage")
plt.tight_layout()
plt.show()

In [ ]:
s2_cells_gdf_w_state = (
    s2_cells_gdf.sjoin(
        combined_gdf[["State Name", "geometry"]], how="inner", predicate="intersects"
    )
    .drop(columns="index_right")
    .drop_duplicates()
)

In [ ]:
# note: this will have duplicate s2 cell rows with different state names if the s2 cell overlaps multiple states
# this is expected and is required for the next steps logic to work correctly
s2_cells_gdf_w_state

#### Download the S2 cells

In [ ]:
download_VIDA_rooftops_data_by_s2(s2_cell_ids, "IND", RAW_DATA_DIR / "02. Rooftop Data")

## 6. Load rooftops and match to areas

In [ ]:
combined_gdf

In [ ]:
state_names = combined_gdf["State Name"].sort_values().unique()

for state_name in tqdm(state_names):
    print(f"Processing state: {state_name}")

    # Filter the s2 cells and rooftops gdf to the current state
    s2_cell_ids = set(
        s2_cells_gdf_w_state[s2_cells_gdf_w_state["State Name"] == state_name][
            "s2_cell_id"
        ]
    )
    print(
        f"Number of S2 cells that overlap our shapes in {state_name}: {len(s2_cell_ids)}"
    )
    gdf_subset = combined_gdf[combined_gdf["State Name"] == state_name]

    # Get matched rooftops for each S2 cell in the state
    matched_rooftop_centroids_gdf_list = []
    for s2_cell_id in tqdm(s2_cell_ids):
        matched_rooftop_centroids_gdf = get_matched_rooftop_centroids_from_s2_file(
            s2_file_dir=RAW_DATA_DIR / "02. Rooftop Data",
            s2_cell_id=s2_cell_id,
            boundaries_gdf=gdf_subset,
        )
        matched_rooftop_centroids_gdf_list.append(matched_rooftop_centroids_gdf)
    matched_rooftop_centroids_gdf = pd.concat(
        matched_rooftop_centroids_gdf_list, ignore_index=True
    )
    matched_rooftop_centroids_gdf["State Name"] = state_name

    # Save the matched rooftops data
    save_shapefiles(
        matched_rooftop_centroids_gdf,
        CLEANED_DATA_DIR / "01. Matched Rooftop Data" / f"{state_name}",
        "matched_rooftops",
        ["parquet"],
    )

In [ ]:
# ax = matched_rooftop_centroids_gdf.sample(1000).plot(
#     cmap=ListedColormap(generate_colormap(len(matched_rooftop_centroids_gdf))),
# )
# gdf_subset.plot(ax=ax, color="none", edgecolor="black", linewidth=0.5)

## 7. Load matched rooftops

In [ ]:
matched_rooftop_dir = CLEANED_DATA_DIR / "01. Matched Rooftop Data"
all_filepaths = list(matched_rooftop_dir.glob("**/*.parquet"))
all_filepaths = [f for f in all_filepaths if f.is_file()]

# # filter to those that have selected_states in the name
# all_filepaths = [
#     f for f in all_filepaths if any(state in f.parent.name for state in selected_states)
# ]

# load all shapes into one gdf
matched_rooftops_gdf_list = []
for filepath in tqdm(all_filepaths):
    matched_rooftops_gdf_list.append(gpd.read_parquet(filepath))
matched_rooftops_gdf = gpd.GeoDataFrame(
    pd.concat(matched_rooftops_gdf_list, ignore_index=True)
).to_crs(4326)

In [ ]:
len(matched_rooftops_gdf)

In [ ]:
no_rooftop_PSU_IDs = set(combined_gdf["PSU ID"].unique()).difference(
    set(matched_rooftops_gdf["PSU ID"].unique())
)
no_rooftop_PSU_gdf = combined_gdf[combined_gdf["PSU ID"].isin(no_rooftop_PSU_IDs)]
no_rooftop_PSU_gdf

In [ ]:
save_shapefiles(
    no_rooftop_PSU_gdf,
    OUTPUT_DATA_DIR / "Sampled PSUs",
    "PSUs_with_no_rooftops",
    ["csv", "kml"],
)

## 8. Sample rooftops

In [ ]:
# Define the base number of rooftops per ward
ROOFTOPS_PER_WARD = 75

# Sample rooftops, multiplying by Ward Count for each PSU.
sampled_rooftops = matched_rooftops_gdf.groupby("PSU ID", group_keys=False).apply(
    lambda x: x.sample(
        n=min(ROOFTOPS_PER_WARD * int(x["Ward Count"].iloc[0]), x.shape[0]),
        random_state=42,
    )
)

In [ ]:
print("Length of matched_rooftops_gdf:", len(matched_rooftops_gdf))
print("Length of sampled rooftops:", len(sampled_rooftops))

In [ ]:
# Check if Ward Count is correctly influencing sample sizes
TEMP_ward_count_df = matched_rooftops_gdf[["PSU ID", "Ward Count"]].drop_duplicates()
TEMP_ward_count_df["Expected Rooftop Count"] = (
    TEMP_ward_count_df["Ward Count"] * ROOFTOPS_PER_WARD
)
TEMP_sampled_counts = (sampled_rooftops.groupby("PSU ID").size()).reset_index(
    name="Sampled Rooftop Count"
)

# Merge the two dataframes
TEMP_check_df = TEMP_ward_count_df.merge(TEMP_sampled_counts, on="PSU ID")
TEMP_check_df["Rooftop Count Difference"] = (
    TEMP_check_df["Expected Rooftop Count"] - TEMP_check_df["Sampled Rooftop Count"]
)
TEMP_check_df[TEMP_check_df["Rooftop Count Difference"] != 0]

In [ ]:
sampled_rooftops.plot(
    figsize=(8, 8),
    column="State Name",
    cmap="tab20",
    edgecolor="black",
    linewidth=0.5,
    legend=True,
)

### Add sample-level rooftop numbering ID columns

In [ ]:
# Rooftop number within each state
sampled_rooftops["Rooftop State ID"] = (
    sampled_rooftops.groupby("State Name").cumcount() + 1
)

# Rooftop number within each PSU ID
sampled_rooftops["Rooftop PSU ID"] = sampled_rooftops.groupby("PSU ID").cumcount() + 1
# add prefix of "PIN "  to the Rooftop PSU ID
sampled_rooftops["Rooftop PSU ID"] = "PIN " + sampled_rooftops["Rooftop PSU ID"].astype(
    str
)

# Rooftop unique ID
sampled_rooftops["Rooftop Unique ID"] = sampled_rooftops.apply(
    lambda row: f"STATE_{row['Rooftop State ID']}_PSU_ID_{row['PSU ID']}_ROOFTOP_{row['Rooftop PSU ID']}",
    axis=1,
)

### Add gmap link

In [ ]:
sampled_rooftops["latitude_original"] = sampled_rooftops.geometry.y
sampled_rooftops["longitude_original"] = sampled_rooftops.geometry.x
sampled_rooftops["gmap_link_original"] = create_gmap_links(
    df=sampled_rooftops,
    lat_name="latitude_original",
    lon_name="longitude_original",
)

### Select only useful columns

**Required columns:**
- PSU info
    - Unique ID across all rooftops
    - Rooftop state ID, #
    - Rooftop PSU ID, #

    - PSU Unit: Ward, TV, Subdistrict
    - PSU sample size

- geospatial info
    - google maps link
    - coordinates
    - geometry

- Admin location info
    - State code and name
    - District code and Name
    - Subdistrict code and name
    - TV code and name
    - Ward code and name

In [ ]:
chosen_cols = [
    ## IDs
    "Rooftop State ID",
    "Rooftop PSU ID",
    "Rooftop Unique ID",
    ## Geospatial data
    "geometry",
    "latitude_original",
    "longitude_original",
    "gmap_link_original",
    ## PSU info
    "PSU ID",
    "PSU Type",
    "Ward Count",
    ## Location info
    "State Code",
    "State Name",
    "State Changed",
    "District Code",
    "District Name",
    "Subdistrict Code",
    "Subdistrict Name",
    "TV Code",
    "TV Name",  # (from MapSolve)
    "Ward Code",
    "Ward/Village Name",
    "Urban/Rural",
    "PCA_ID",  # combined TVCode-WardCode
    "Ward Population",
    "Subdistrict Population",
    "State Population",
    # "Complete ID",
    ## Admin information
    # "Included in Panel",
    "Ward Boundary Available with MapSolve",
    # "State Shared by MapSolve",
    "Ward Boundary Given",
    "TV Boundary Given",
    "SubDistrict Boundary Given",
    "Delivery State",
    # "UID",
    # "s2_rooftop_id",
    ## MapSolve location info
    "State Code_MapSolve",
    "State Name_MapSolve",
    "District Code_MapSolve",
    "District Name_MapSolve",
    "Subdistrict Code_MapSolve",
    "Subdistrict Name_MapSolve",
    "TV Code_MapSolve",
    # "TV Name",
    "Ward Code_MapSolve",
    "PCA_ID_MapSolve",
    "Total Population",
    # ## rooftop info
    # "boundary_id",
    # "bf_source",
    # "confidence",
    # "area_in_meters",
    # "s2_id",
    # "country_iso",
    # "geohash",
    # "bbox",
]

In [ ]:
sampled_rooftops_organised_gdf = sampled_rooftops[chosen_cols]

In [ ]:
sampled_rooftops_organised_gdf.rename(
    columns={
        "TV Name": "TV Name_MapSolve",
        "PSU ID": "PSU ID",
        "Total Population": "PSU Total Population_MapSolve",
    },
    inplace=True,
)

In [ ]:
# set Ward Codes of 0.0 to NaN
sampled_rooftops_organised_gdf.loc[
    sampled_rooftops_organised_gdf["Ward Code"] == 0.0, "Ward Code"
] = np.nan

In [ ]:
sampled_rooftops_organised_gdf

### Save sampled data (original rooftop pins)

In [ ]:
save_shapefiles(
    sampled_rooftops_organised_gdf,
    OUTPUT_DATA_DIR / "01. Sampled Rooftop Data",
    "sampled_rooftops_centroids_original",
    ["csv", "parquet"],
)

save_shapefiles(
    sampled_rooftops_organised_gdf.drop(columns=["gmap_link_original"]),
    OUTPUT_DATA_DIR / "01. Sampled Rooftop Data",
    "sampled_rooftops_centroids_original",
    ["kml"],
)

## 9. Snap points to road

In [ ]:
import yaml
from shapely import Point

from utils import (
    get_nearest_points_on_road_batch,
    get_nearest_points_on_road_batch_parallel,
)

In [ ]:
# load API key
with open("../secrets/api_keys.yaml", "r") as f:
    config = yaml.safe_load(f)
    api_key = config["GOOGLE_ROADS_API_KEY"]

In [ ]:
get_nearest_points_on_road_batch(
    [Point(77.11432151622034, 28.677391409999522)], api_key
)

In [ ]:
# test
get_nearest_points_on_road_batch(
    sampled_rooftops_organised_gdf.geometry.iloc[:5], api_key
)

In [ ]:
get_nearest_points_on_road_batch_parallel(
    sampled_rooftops_organised_gdf.iloc[:150], api_key
)

#### Snap points to road

In [ ]:
snapped_points_series = get_nearest_points_on_road_batch_parallel(
    sampled_rooftops_organised_gdf, api_key, max_workers=12
)
# took 1 second for 1,600 points (Panel)
# 30s for all 53,000 points

In [ ]:
sampled_rooftops_snapped_gdf = sampled_rooftops_organised_gdf.copy()
sampled_rooftops_snapped_gdf["geometry_snapped"] = (
    sampled_rooftops_snapped_gdf.index.map(snapped_points_series)
)

In [ ]:
# Make new Geometry Type column which has values "Original" or "Snapped to Road"
sampled_rooftops_snapped_gdf["Geometry Type"] = (
    sampled_rooftops_snapped_gdf["geometry_snapped"]
    .notna()
    .replace({True: "Snapped to Road", False: "Original"})
)
sampled_rooftops_snapped_gdf["Geometry Type"].value_counts()

#### Replace geometry to snapped one (missing filled in with original)

In [ ]:
# backup the original geometry
sampled_rooftops_snapped_gdf["geometry_original"] = sampled_rooftops_snapped_gdf[
    "geometry"
]
# replace the original geometry with the snapped geometry
sampled_rooftops_snapped_gdf["geometry"] = sampled_rooftops_snapped_gdf[
    "geometry_snapped"
]
# drop the snapped geometry column
sampled_rooftops_snapped_gdf = sampled_rooftops_snapped_gdf.drop(
    columns=["geometry_snapped"]
)
# fill in NaN values in the snapped geometry with the original geometry
sampled_rooftops_snapped_gdf["geometry"] = sampled_rooftops_snapped_gdf[
    "geometry"
].fillna(sampled_rooftops_snapped_gdf["geometry_original"])

In [ ]:
sampled_rooftops_snapped_gdf["geometry"].isna().sum()

#### Update lat, lon, gmap_link

In [ ]:
sampled_rooftops_snapped_gdf["latitude"] = list(sampled_rooftops_snapped_gdf.geometry.y)
sampled_rooftops_snapped_gdf["longitude"] = list(
    sampled_rooftops_snapped_gdf.geometry.x
)
sampled_rooftops_snapped_gdf["gmap_link"] = create_gmap_links(
    df=sampled_rooftops_snapped_gdf,
    lat_name="latitude",
    lon_name="longitude",
)

#### Reorganise

In [ ]:
sampled_rooftops_snapped_gdf = sampled_rooftops_snapped_gdf[
    [
        "Rooftop State ID",
        "Rooftop PSU ID",
        "Rooftop Unique ID",
        # new columns start
        "Geometry Type",
        "geometry",
        "latitude",
        "longitude",
        "gmap_link",
        # new columns end
        "geometry_original",
        "latitude_original",
        "longitude_original",
        "gmap_link_original",
        "PSU ID",
        "PSU Type",
        "Ward Count",
        "State Code",
        "State Name",
        "State Changed",
        "District Code",
        "District Name",
        "Subdistrict Code",
        "Subdistrict Name",
        "TV Code",
        "TV Name_MapSolve",
        "Ward Code",
        "Ward/Village Name",
        "Urban/Rural",
        "PCA_ID",
        "Ward Population",
        "Subdistrict Population",
        "State Population",
        # "Included in Panel",
        "Ward Boundary Available with MapSolve",
        "Ward Boundary Given",
        "TV Boundary Given",
        "SubDistrict Boundary Given",
        "Delivery State",
        "State Code_MapSolve",
        "State Name_MapSolve",
        "District Code_MapSolve",
        "District Name_MapSolve",
        "Subdistrict Code_MapSolve",
        "Subdistrict Name_MapSolve",
        "TV Code_MapSolve",
        "Ward Code_MapSolve",
        "PCA_ID_MapSolve",
        "PSU Total Population_MapSolve",
    ]
]

#### Make lines between original and snapped points

In [ ]:
from shapely.geometry import LineString

In [ ]:
sampled_rooftops_snapped_gdf["geometry_line"] = sampled_rooftops_snapped_gdf.apply(
    lambda row: LineString([row["geometry_original"], row["geometry"]]), axis=1
)

In [ ]:
sampled_rooftops_snapped_gdf["geometry_line"].isna().sum()

#### Save new files: snapped points, snapped lines

In [ ]:
# Save CSV and parquet
save_shapefiles(
    sampled_rooftops_snapped_gdf.drop(
        columns=[
            "geometry_original",
            "geometry_line",
        ]
    ),
    OUTPUT_DATA_DIR / "01. Sampled Rooftop Data",
    "sampled_rooftops_snapped_points",
    ["csv", "parquet"],
)

In [ ]:
# Save KML
save_shapefiles(
    sampled_rooftops_snapped_gdf.drop(
        columns=[
            "geometry_original",
            "geometry_line",
            # bad cols for KML
            "gmap_link_original",
            "gmap_link",
        ]
    ),
    OUTPUT_DATA_DIR / "01. Sampled Rooftop Data",
    "sampled_rooftops_snapped_points",
    ["kml"],
)

In [ ]:
# Save lines
sampled_rooftops_line_gdf = sampled_rooftops_snapped_gdf.copy()
sampled_rooftops_line_gdf["geometry"] = sampled_rooftops_line_gdf["geometry_line"]
sampled_rooftops_line_gdf = sampled_rooftops_line_gdf.drop(
    columns=["geometry_original", "geometry_line"]
)

save_shapefiles(
    sampled_rooftops_line_gdf.drop(
        # drop kml unfriendly columns
        columns=["gmap_link_original", "gmap_link"]
    ),
    OUTPUT_DATA_DIR / "01. Sampled Rooftop Data",
    "sampled_rooftops_snapped_lines",
    ["parquet", "kml"],
)

## Save per state

In [ ]:
for state in tqdm(sampled_rooftops_organised_gdf["State Name"].unique()):
    state_output_folder = OUTPUT_DATA_DIR / "01. Sampled Rooftop Data" / state

    # save original points
    selected_state_original_gdf = sampled_rooftops_organised_gdf[
        sampled_rooftops_organised_gdf["State Name"] == state
    ]

    save_shapefiles(
        selected_state_original_gdf,
        state_output_folder,
        f"{state}_sampled_rooftops_centroids_original",
        ["csv", "parquet"],
    )
    save_shapefiles(
        selected_state_original_gdf.drop(
            # drop kml unfriendly columns
            columns=["gmap_link_original"]
        ),
        state_output_folder,
        f"{state}_sampled_rooftops_centroids_original",
        ["kml"],
    )

    # Save snapped points
    selected_state_snapped_gdf = sampled_rooftops_snapped_gdf[
        sampled_rooftops_snapped_gdf["State Name"] == state
    ].drop(
        columns=[
            "geometry_original",
            "geometry_line",
        ]
    )
    save_shapefiles(
        selected_state_snapped_gdf,
        state_output_folder,
        f"{state}_sampled_rooftops_snapped_points",
        ["csv", "parquet"],
    )
    save_shapefiles(
        selected_state_snapped_gdf.drop(
            # drop kml unfriendly columns
            columns=["gmap_link_original", "gmap_link"]
        ),
        state_output_folder,
        f"{state}_sampled_rooftops_snapped_points",
        ["kml"],
    )

    # Save lines
    selected_state_sampled_rooftops_line_gdf = sampled_rooftops_line_gdf[
        sampled_rooftops_line_gdf["State Name"] == state
    ]
    save_shapefiles(
        selected_state_sampled_rooftops_line_gdf.drop(
            # drop kml unfriendly columns
            columns=["gmap_link_original", "gmap_link"]
        ),
        state_output_folder,
        f"{state}_sampled_rooftops_snapped_lines",
        ["parquet", "kml"],
    )

## Sidequests

### Break up files

In [14]:
rows_per_part = 1600

In [15]:
lines_gdf = gpd.read_parquet(
    OUTPUT_DATA_DIR
    / "01. Sampled Rooftop Data"
    / "sampled_rooftops_snapped_lines.parquet"
)

In [16]:
lines_gdf

,Rooftop State ID,Rooftop PSU ID,Rooftop Unique ID,Geometry Type,geometry,latitude,longitude,latitude_original,longitude_original,PSU ID,...,State Code_MapSolve,State Name_MapSolve,District Code_MapSolve,District Name_MapSolve,Subdistrict Code_MapSolve,Subdistrict Name_MapSolve,TV Code_MapSolve,Ward Code_MapSolve,PCA_ID_MapSolve,PSU Total Population_MapSolve
777460,1,PIN 1,STATE_1_PSU_ID_WARD_801794-1_ROOFTOP_PIN 1,Snapped to Road,"LINESTRING (85.31096 23.39589, 85.31110 23.39591)",23.395907,85.311095,23.395887,85.310964,WARD_801794-1,...,20.0,Jharkhand,364.0,Ranchi,2684.0,Kanke,801794.0,1.0,None,9579.0
780913,2,PIN 2,STATE_2_PSU_ID_WARD_801794-1_ROOFTOP_PIN 2,Snapped to Road,"LINESTRING (85.31659 23.42254, 85.31659 23.42257)",23.422572,85.316588,23.422539,85.316586,WARD_801794-1,...,20.0,Jharkhand,364.0,Ranchi,2684.0,Kanke,801794.0,1.0,None,9579.0
779052,3,PIN 3,STATE_3_PSU_ID_WARD_801794-1_ROOFTOP_PIN 3,Snapped to Road,"LINESTRING (85.30983 23.41015, 85.30990 23.41031)",23.410313,85.309896,23.410149,85.309832,WARD_801794-1,...,20.0,Jharkhand,364.0,Ranchi,2684.0,Kanke,801794.0,1.0,None,9579.0
780151,4,PIN 4,STATE_4_PSU_ID_WARD_801794-1_ROOFTOP_PIN 4,Snapped to Road,"LINESTRING (85.31163 23.41675, 85.31156 23.41675)",23.416750,85.311561,23.416745,85.311627,WARD_801794-1,...,20.0,Jharkhand,364.0,Ranchi,2684.0,Kanke,801794.0,1.0,None,9579.0
779453,5,PIN 5,STATE_5_PSU_ID_WARD_801794-1_ROOFTOP_PIN 5,Snapped to Road,"LINESTRING (85.31369 23.41092, 85.31371 23.41091)",23.410911,85.313710,23.410915,85.313689,WARD_801794-1,...,20.0,Jharkhand,364.0,Ranchi,2684.0,Kanke,801794.0,1.0,None,9579.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
425599,6971,PIN 71,STATE_6971_PSU_ID_WARD_802035-9_ROOFTOP_PIN 71,Snapped to Road,"LINESTRING (81.72122 21.19236, 81.72092 21.19282)",21.192825,81.720922,21.192355,81.721218,WARD_802035-9,...,22.0,Chhattisgarh,410.0,Raipur,3332.0,Raipur,802035.0,9.0,None,1094.0
425732,6972,PIN 72,STATE_6972_PSU_ID_WARD_802035-9_ROOFTOP_PIN 72,Snapped to Road,"LINESTRING (81.72163 21.19551, 81.72162 21.19543)",21.195427,81.721618,21.195506,81.721634,WARD_802035-9,...,22.0,Chhattisgarh,410.0,Raipur,3332.0,Raipur,802035.0,9.0,None,1094.0
425576,6973,PIN 73,STATE_6973_PSU_ID_WARD_802035-9_ROOFTOP_PIN 73,Snapped to Road,"LINESTRING (81.72029 21.19246, 81.71999 21.19247)",21.192473,81.719990,21.192465,81.720292,WARD_802035-9,...,22.0,Chhattisgarh,410.0,Raipur,3332.0,Raipur,802035.0,9.0,None,1094.0
425604,6974,PIN 74,STATE_6974_PSU_ID_WARD_802035-9_ROOFTOP_PIN 74,Snapped to Road,"LINESTRING (81.71994 21.19273, 81.72000 21.19273)",21.192726,81.720000,21.192729,81.719937,WARD_802035-9,...,22.0,Chhattisgarh,410.0,Raipur,3332.0,Raipur,802035.0,9.0,None,1094.0


In [17]:
# split into parts and save each part
num_parts = len(lines_gdf) // rows_per_part + 1

line_output_dir = OUTPUT_DATA_DIR / "01. Sampled Rooftop Data" / "Split Lines"
for i in range(num_parts):
    start_idx = i * rows_per_part
    end_idx = min((i + 1) * rows_per_part, len(lines_gdf))
    part_gdf = lines_gdf.iloc[start_idx:end_idx]

    save_shapefiles(
        part_gdf,
        line_output_dir,
        f"sampled_rooftops_snapped_lines_part_{i}",
        ["kml"],
    )

/Users/amiremami/.pyenv/versions/geospatial/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Value 'WARD_801794-1' of field sampled_rooftops_snapped_lines_part_0.latitude_original parsed incompletely to real 0.
  ogr_write(
/Users/amiremami/.pyenv/versions/geospatial/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Value 'ward' of field sampled_rooftops_snapped_lines_part_0.longitude_original parsed incompletely to real 0.
  ogr_write(
/Users/amiremami/.pyenv/versions/geospatial/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Value 'No' of field sampled_rooftops_snapped_lines_part_0.State Code parsed incompletely to real 0.
  ogr_write(
/Users/amiremami/.pyenv/versions/geospatial/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Value 'Urban' of field sampled_rooftops_snapped_lines_part_0.Ward Code parsed incompletely to real 0.
  ogr_write(
/Users/amiremami/.pyenv/versions/geospatial/lib/python3.12/site-packages/pyogrio/r

In [18]:
# do the above but per state
for state in tqdm(lines_gdf["State Name"].unique()):
    state_output_folder = (
        OUTPUT_DATA_DIR / "01. Sampled Rooftop Data" / state / "Split Lines"
    )

    selected_state_lines_gdf = lines_gdf[lines_gdf["State Name"] == state]
    num_parts = len(selected_state_lines_gdf) // rows_per_part + 1

    for i in range(num_parts):
        start_idx = i * rows_per_part
        end_idx = min((i + 1) * rows_per_part, len(selected_state_lines_gdf))
        part_gdf = selected_state_lines_gdf.iloc[start_idx:end_idx]

        save_shapefiles(
            part_gdf,
            state_output_folder,
            f"sampled_rooftops_snapped_lines_part_{i}",
            ["kml"],
        )

  0%|          | 0/3 [00:00<?, ?it/s]

/Users/amiremami/.pyenv/versions/geospatial/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Value 'WARD_801794-1' of field sampled_rooftops_snapped_lines_part_0.latitude_original parsed incompletely to real 0.
  ogr_write(
/Users/amiremami/.pyenv/versions/geospatial/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Value 'ward' of field sampled_rooftops_snapped_lines_part_0.longitude_original parsed incompletely to real 0.
  ogr_write(
/Users/amiremami/.pyenv/versions/geospatial/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Value 'No' of field sampled_rooftops_snapped_lines_part_0.State Code parsed incompletely to real 0.
  ogr_write(
/Users/amiremami/.pyenv/versions/geospatial/lib/python3.12/site-packages/pyogrio/raw.py:723: RuntimeWarning: Value 'Urban' of field sampled_rooftops_snapped_lines_part_0.Ward Code parsed incompletely to real 0.
  ogr_write(
/Users/amiremami/.pyenv/versions/geospatial/lib/python3.12/site-packages/pyogrio/r

#### Do the same thing as above but for the snapped points CSV

In [19]:
snapped_points_gdf = gpd.read_parquet(
    OUTPUT_DATA_DIR
    / "01. Sampled Rooftop Data"
    / "sampled_rooftops_snapped_points.parquet"
)

In [20]:
num_parts = len(snapped_points_gdf) // rows_per_part + 1
snapped_points_output_dir = (
    OUTPUT_DATA_DIR / "01. Sampled Rooftop Data" / "Snapped Points Split"
)
for i in range(num_parts):
    start_idx = i * rows_per_part
    end_idx = min((i + 1) * rows_per_part, len(snapped_points_gdf))
    part_gdf = snapped_points_gdf.iloc[start_idx:end_idx]
    snapped_points_output_dir.mkdir(parents=True, exist_ok=True)

    save_shapefiles(
        part_gdf,
        snapped_points_output_dir,
        f"sampled_rooftops_snapped_points_part_{i}",
        ["csv"],
    )

In [21]:
# do the above but per state
for state in tqdm(snapped_points_gdf["State Name"].unique()):
    state_output_folder = (
        OUTPUT_DATA_DIR / "01. Sampled Rooftop Data" / state / "Split Snapped Points"
    )

    selected_state_snapped_points_gdf = snapped_points_gdf[
        snapped_points_gdf["State Name"] == state
    ]
    num_parts = len(selected_state_snapped_points_gdf) // rows_per_part + 1

    for i in range(num_parts):
        start_idx = i * rows_per_part
        end_idx = min((i + 1) * rows_per_part, len(selected_state_snapped_points_gdf))
        part_gdf = selected_state_snapped_points_gdf.iloc[start_idx:end_idx]

        save_shapefiles(
            part_gdf,
            state_output_folder,
            f"sampled_rooftops_snapped_points_part_{i}",
            ["csv"],
        )

  0%|          | 0/3 [00:00<?, ?it/s]